In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder

In [30]:
employees = pd.read_csv("../datasets/employee_cleaned.csv")
regions = pd.read_csv("../region_benefit_profiles.csv")

In [31]:
print(employees.shape)
print(regions.shape)

employees.head()

(10008, 17)
(4, 9)


,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,last_contact_date,last_contact_channel,plan_tier_requested,broker_channel,prior_year_enrolled,legacy_propensity_score,outreach_notes
0,12324,28,Male,Divorced,44047.60,Full-time,West,No,1.4,0,2024-03-30,SMS,Basic,Employer-Sponsored,0,0.104,Spouse covered elsewhere
1,17825,23,Male,Married,80111.28,Full-time,Midwest,Yes,0.5,1,2024-04-18,Phone,Standard,Direct,-1,0.874,Requested more time
2,15200,39,Male,Married,69855.65,Full-time,South,Yes,3.4,1,2024-06-09,NaN,Silver,Direct,0,0.870,No response to first outreach
3,16690,31,Female,Married,91567.33,Full-time,South,No,1.2,1,2024-07-29,Email,Premium,Employer-Sponsored,1,0.894,Requested more time
4,17465,42,Female,Single,59861.68,Contract,Midwest,Yes,20.0,0,2024-05-06,Email,Standard,Employer-Sponsored,-1,0.163,Requested more time


In [32]:
employees["last_contact_date"] = pd.to_datetime(
    employees["last_contact_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print(employees["last_contact_date"].dtype)

datetime64[ns]


In [33]:
employees[["last_contact_date"]].head(10)

,last_contact_date
0,2024-03-30
1,2024-04-18
2,2024-06-09
3,2024-07-29
4,2024-05-06
5,2024-01-12
6,2024-07-13
7,2024-08-01
8,2024-05-16
9,2024-05-26


In [34]:
employees["contact_year"] = employees["last_contact_date"].dt.year
employees["contact_month"] = employees["last_contact_date"].dt.month
employees["contact_day"] = employees["last_contact_date"].dt.day
employees["contact_dayofweek"] = employees["last_contact_date"].dt.dayofweek

In [35]:
employees[
    [
        "last_contact_date",
        "contact_year",
        "contact_month",
        "contact_day",
        "contact_dayofweek",
    ]
].head()

,last_contact_date,contact_year,contact_month,contact_day,contact_dayofweek
0,2024-03-30,2024,3,30,5
1,2024-04-18,2024,4,18,3
2,2024-06-09,2024,6,9,6
3,2024-07-29,2024,7,29,0
4,2024-05-06,2024,5,6,0


In [36]:
employees.drop(columns=["last_contact_date"], inplace=True)

In [37]:
df = employees.merge(
    regions,
    on="region",
    how="left"
)

In [38]:
print(df.shape)

df.head()

(10008, 28)


,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,...,contact_day,contact_dayofweek,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,12324,28,Male,Divorced,44047.60,Full-time,West,No,1.4,0,...,30,5,2582,0.613,65007.07,574,4.7,437,28,Low
1,17825,23,Male,Married,80111.28,Full-time,Midwest,Yes,0.5,1,...,18,3,2488,0.617,64921.46,595,4.4,469,18,High
2,15200,39,Male,Married,69855.65,Full-time,South,Yes,3.4,1,...,9,6,2424,0.628,65200.49,481,3.3,324,17,Med
3,16690,31,Female,Married,91567.33,Full-time,South,No,1.2,1,...,29,0,2424,0.628,65200.49,481,3.3,324,17,Med
4,17465,42,Female,Single,59861.68,Contract,Midwest,Yes,20.0,0,...,6,0,2488,0.617,64921.46,595,4.4,469,18,High


In [39]:
print(df["employee_id"].duplicated().sum())

df.drop_duplicates(subset="employee_id", inplace=True)

print(df.shape)

8
(10000, 28)


In [40]:
df["premium_salary_percentage"] = (
    df["avg_premium_cost_usd"] / df["salary"]
) * 100

df["outreach_capacity_ratio"] = (
    df["hr_outreach_capacity"] /
    df["n_employees_region"]
)

df["salary_diff_region"] = (
    df["salary"] -
    df["avg_salary_region"]
)

In [41]:
df.drop(
    columns=[
        "employee_id",
        "legacy_propensity_score",
        "hist_enrollment_rate_region",
        "outreach_notes",
        
    ],
    inplace=True,
    errors="ignore",
)

In [42]:


categorical_cols = [
    "gender",
    "marital_status",
    "employment_type",
    "region",
    "has_dependents",
    "last_contact_channel",
    "plan_tier_requested",
    "broker_channel",
    "state_mandate_level",
]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

df[categorical_cols] = encoder.fit_transform(
    df[categorical_cols].astype(str)
)

In [43]:
df[categorical_cols].head()

,gender,marital_status,employment_type,region,has_dependents,last_contact_channel,plan_tier_requested,broker_channel,state_mandate_level
0,1.0,0.0,1.0,3.0,0.0,2.0,0.0,1.0,1.0
1,1.0,1.0,1.0,0.0,1.0,1.0,5.0,0.0,0.0
2,1.0,1.0,1.0,2.0,1.0,3.0,4.0,0.0,2.0
3,0.0,1.0,1.0,2.0,0.0,0.0,3.0,1.0,2.0
4,0.0,2.0,0.0,0.0,1.0,0.0,5.0,1.0,0.0


In [44]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10007
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          10000 non-null  int64  
 1   gender                       10000 non-null  float64
 2   marital_status               10000 non-null  float64
 3   salary                       10000 non-null  float64
 4   employment_type              10000 non-null  float64
 5   region                       10000 non-null  float64
 6   has_dependents               10000 non-null  float64
 7   tenure_years                 10000 non-null  float64
 8   enrolled                     10000 non-null  int64  
 9   last_contact_channel         10000 non-null  float64
 10  plan_tier_requested          10000 non-null  float64
 11  broker_channel               10000 non-null  float64
 12  prior_year_enrolled          10000 non-null  int64  
 13  contact_year         

In [45]:
df.to_csv(
    "../datasets/final_merged.csv",
    index=False
)